# EDA — População do IBGE

A Bronze preserva todas as linhas da aba `Municípios`, incluindo título, cabeçalho, notas e linhas vazias. O título da fonte informa a referência de **1º de julho de 2025**. A data de atualização no nome do arquivo não substitui essa referência.

**Perguntas:**

1. Quais linhas representam municípios e quais são documentação da planilha?
2. Existem códigos, nomes ou populações ausentes, inválidos ou duplicados?
3. Conseguimos construir a chave IBGE sem perder os zeros iniciais do código municipal?
4. Todas as 27 capitais da nossa referência estão presentes?
5. Como a população se distribui no conjunto completo e no recorte das capitais?

As conversões e seleções desta EDA são temporárias. Não há consultas a APIs, correção automática, remoção de registros da Bronze ou gravação da Silver.


## 1. Bibliotecas e caminhos

Usamos Spark para as verificações e Matplotlib para os gráficos. Não há dependência do pandas. O notebook deve ser executado com o kernel do ambiente virtual, a partir de uma pasta dentro do projeto.


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType


In [ ]:
project_directory = Path.cwd().resolve()

while (
    not (project_directory / "pipeline" / "bronze").is_dir()
    or not (project_directory / "data").is_dir()
):

    if project_directory == project_directory.parent:
        raise FileNotFoundError("Execute o notebook a partir de uma pasta dentro do projeto.")

    project_directory = project_directory.parent

BRONZE_DIRECTORY = project_directory / "data" / "bronze"
POPULATION_DIRECTORY = BRONZE_DIRECTORY / "ibge" / "population"
MUNICIPALITIES_DIRECTORY = BRONZE_DIRECTORY / "ibge" / "municipalities"
local_filesystem_jar = (
    project_directory / "pipeline" / "bronze" / "lib" / "hadoop-bare-naked-local-fs-0.1.0.jar"
)

for required_path in [POPULATION_DIRECTORY, MUNICIPALITIES_DIRECTORY, local_filesystem_jar]:

    if not required_path.exists():
        raise FileNotFoundError(f"Caminho necessário não encontrado: {required_path}")

print(f"População: {POPULATION_DIRECTORY}")
print(f"Referência das capitais: {MUNICIPALITIES_DIRECTORY}")


## 2. Sessão Spark

Mantemos a configuração local já validada, com duas tarefas simultâneas. Use um kernel novo para aplicar o classpath desde a criação do Spark.


In [ ]:
os.environ["PYSPARK_PYTHON"] = sys.executable

spark = (
    SparkSession.builder.appName("Viajando com o dado - EDA Populacao IBGE")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.driver.extraClassPath", str(local_filesystem_jar))
    .config("spark.executor.extraClassPath", str(local_filesystem_jar))
    .config("spark.hadoop.fs.file.impl", "com.globalmentor.apache.hadoop.fs.BareLocalFileSystem")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print(f"Versão do Spark: {spark.version}")


## 3. Estrutura da Bronze e rastreabilidade

Cada registro representa uma linha da planilha. `row_values_json` guarda as células, e `source_file`, `source_sheet` e `source_row_number` permitem localizar sua origem.


In [ ]:
population_rows = (
    spark.read.option("recursiveFileLookup", True)
    .option("pathGlobFilter", "*.parquet")
    .parquet(POPULATION_DIRECTORY.resolve().as_posix())
    .cache()
)

municipalities = (
    spark.read.option("recursiveFileLookup", True)
    .option("pathGlobFilter", "*.parquet")
    .parquet(MUNICIPALITIES_DIRECTORY.resolve().as_posix())
)

raw_row_count = population_rows.count()
print(f"Linhas preservadas na Bronze: {raw_row_count}")

population_rows.printSchema()
population_rows.orderBy("source_row_number").show(5, truncate=130)
population_rows.groupBy("source_file", "source_sheet").count().show(truncate=False)

repeated_source_rows = (
    population_rows.groupBy("source_file", "source_sheet", "source_row_number")
    .count()
    .filter(F.col("count") > 1)
)

source_duplicate_count = repeated_source_rows.count()

print(f"Identificadores de linha de origem repetidos: {source_duplicate_count}")
repeated_source_rows.show(truncate=False)


## 4. Interpretar as células e classificar as linhas

As seis posições observadas são: UF, código da UF, código municipal, nome, população estimada e uma coluna adicional sem título. Os números do Excel podem aparecer como `11.0` ou `22787.0`; a extração inicial mantém tudo como texto para inspeção.

Uma linha é candidata a município quando tem uma sigla de duas letras ou códigos numéricos acompanhados de nome. Isso também permite investigar candidatos com UF incorreta ou ausente. Título, cabeçalho e linhas vazias são classificados antes dessa regra. As demais linhas ficam disponíveis como notas/metadados.

JSONs não interpretáveis e estruturas com quantidade de células inesperada são contabilizados separadamente.


In [ ]:
parsed_rows = population_rows.withColumn(
    "row_values",
    F.from_json(F.col("row_values_json"), ArrayType(StringType(), True)),
)

parsed_rows = parsed_rows.withColumn("state_code_raw", F.trim(F.get("row_values", 0)))
parsed_rows = parsed_rows.withColumn("state_id_raw", F.trim(F.get("row_values", 1)))
parsed_rows = parsed_rows.withColumn("municipality_code_raw", F.trim(F.get("row_values", 2)))
parsed_rows = parsed_rows.withColumn("municipality_name", F.trim(F.get("row_values", 3)))
parsed_rows = parsed_rows.withColumn("population_raw", F.trim(F.get("row_values", 4)))
parsed_rows = parsed_rows.withColumn("extra_cell", F.get("row_values", 5))

has_values = F.lit(False)

for cell_index in range(6):

    cell_text = F.coalesce(F.get("row_values", cell_index), F.lit(""))
    has_values = has_values | (F.length(F.trim(cell_text)) > 0)

integer_pattern = r"^[+-]?[0-9]+(?:\.0+)?$"
is_header = (F.col("state_code_raw") == "UF") & (F.col("state_id_raw") == "COD. UF")

has_numeric_keys = F.col("state_id_raw").rlike(integer_pattern) & F.col(
    "municipality_code_raw"
).rlike(integer_pattern)
has_name = F.length(F.coalesce(F.col("municipality_name"), F.lit(""))) > 0

is_candidate = F.col("state_code_raw").rlike("^[A-Za-z]{2}$") | (has_numeric_keys & has_name)

parsed_rows = parsed_rows.withColumn(
    "row_type",
    F.when(F.col("row_values").isNull(), "unparsed")
    .when(~has_values, "blank")
    .when(is_header, "header")
    .when(F.col("state_code_raw").startswith("ESTIMATIVAS DA POPULA"), "title")
    .when(is_candidate, "municipality")
    .otherwise("note_or_metadata"),
).cache()

municipality_rows = parsed_rows.filter(F.col("row_type") == "municipality")
other_rows = parsed_rows.filter(F.col("row_type") != "municipality")

municipality_row_count = municipality_rows.count()
nonmunicipal_row_count = other_rows.count()

unparsed_row_count = parsed_rows.filter(F.col("row_type") == "unparsed").count()
unexpected_shape_count = parsed_rows.filter(
    F.col("row_values").isNotNull() & (F.size("row_values") != 6)
).count()

parsed_rows.groupBy("row_type").count().orderBy(F.desc("count")).show(truncate=False)

print(f"Candidatos a municípios: {municipality_row_count}")
print(f"Linhas de documentação, vazias ou não interpretadas: {nonmunicipal_row_count}")
print(f"JSONs não interpretados: {unparsed_row_count}")
print(f"Linhas com quantidade de células diferente de seis: {unexpected_shape_count}")

other_rows.select("source_row_number", "row_type", "row_values_json").orderBy(
    "source_row_number"
).show(40, truncate=120)


## 5. Visão numérica e chave IBGE

Aceitamos representações inteiras, como `11`, `11.0` e `00015`. Para população, também reconhecemos notação científica, como `1.2E7`, antes de conferir se o valor é inteiro. Textos não reconhecidos e valores fracionários ficam sem conversão para serem investigados, sem arredondamento automático.

A chave municipal é formada pelo código de UF de dois dígitos e pelo código municipal de cinco dígitos. `lpad` recompõe os zeros à esquerda: `11` + `00015` resulta em `1100015`. Os campos originais e os números de linha continuam disponíveis.

A população deve representar uma contagem inteira positiva. Valores inválidos serão apresentados, não imputados.


In [ ]:
population_view = municipality_rows.withColumn("state_code", F.upper(F.col("state_code_raw")))

population_view = population_view.withColumn(
    "state_id_number",
    F.when(
        F.col("state_id_raw").rlike(integer_pattern),
        F.expr("try_cast(try_cast(state_id_raw AS DECIMAL(20, 0)) AS BIGINT)"),
    ),
)

population_view = population_view.withColumn(
    "municipality_code_number",
    F.when(
        F.col("municipality_code_raw").rlike(integer_pattern),
        F.expr("try_cast(try_cast(municipality_code_raw AS DECIMAL(20, 0)) AS BIGINT)"),
    ),
)

population_view = population_view.withColumn(
    "state_id",
    F.when(F.col("state_id_number").between(10, 99), F.col("state_id_number").cast("string")),
)

population_view = population_view.withColumn(
    "municipality_code",
    F.when(
        F.col("municipality_code_number").between(0, 99999),
        F.lpad(F.col("municipality_code_number").cast("string"), 5, "0"),
    ),
)

population_view = population_view.withColumn(
    "municipality_id",
    F.when(
        F.col("state_id").isNotNull() & F.col("municipality_code").isNotNull(),
        F.concat("state_id", "municipality_code").cast("long"),
    ),
)

# Reconhece também a notação científica que o leitor JSON pode produzir.
population_view = population_view.withColumn(
    "population_number",
    F.expr("try_cast(population_raw AS DECIMAL(38, 18))"),
)

population_view = population_view.withColumn(
    "population_estimate",
    F.when(
        F.col("population_number").isNotNull()
        & (F.col("population_number") == F.floor("population_number")),
        F.expr("try_cast(population_number AS BIGINT)"),
    ),
).cache()

population_view.select(
    "state_code",
    "state_id_raw",
    "municipality_code_raw",
    "municipality_code",
    "municipality_id",
    "municipality_name",
    "population_raw",
    "population_estimate",
).orderBy("source_row_number").show(10, truncate=False)


## 6. Ausências e valores inválidos

Contamos ausências nos campos essenciais dos candidatos a municípios. A coluna adicional vazia da planilha não é um campo obrigatório de população e não será tratada como milhares de erros.

Falhas de conversão aparecem separadamente de campos originalmente vazios. Isso evita confundir um texto preenchido, porém inválido, com uma ausência na fonte.


In [ ]:
essential_columns = [
    "state_code_raw",
    "state_id_raw",
    "municipality_code_raw",
    "municipality_name",
    "population_raw",
]
missing_expressions = []

for column_name in essential_columns:

    missing_condition = F.col(column_name).isNull() | (F.trim(F.col(column_name)) == "")
    missing_expressions.append(F.sum(F.when(missing_condition, 1).otherwise(0)).alias(column_name))

missing_totals = population_view.agg(*missing_expressions).first()
missing_records = []
missing_cell_count = 0

for column_name in essential_columns:

    missing_count = int(missing_totals[column_name] or 0)
    missing_percent = 0.0

    if municipality_row_count > 0:
        missing_percent = round(100.0 * missing_count / municipality_row_count, 2)

    missing_records.append((column_name, missing_count, missing_percent))
    missing_cell_count += missing_count

spark.createDataFrame(
    missing_records, "column_name string, missing_count long, missing_percent double"
).show(truncate=False)

invalid_codes = population_view.filter(F.col("municipality_id").isNull())
invalid_population = population_view.filter(
    F.col("population_estimate").isNull() | (F.col("population_estimate") <= 0)
)
invalid_code_count = invalid_codes.count()
invalid_population_count = invalid_population.count()

print(f"Registros sem chave municipal válida: {invalid_code_count}")
invalid_codes.select(
    "source_row_number",
    "state_code_raw",
    "state_id_raw",
    "municipality_code_raw",
    "municipality_name",
).show(truncate=False)

print(
    f"Populações não inteiras, ausentes, fora do tipo suportado ou não positivas: {invalid_population_count}"
)
invalid_population.select("source_row_number", "municipality_name", "population_raw").show(
    truncate=False
)


## 7. Duplicações, UFs e códigos

Esperamos uma linha por código de município na planilha. Nomes iguais em UFs diferentes não constituem, isoladamente, duplicação.

Conferimos também se cada sigla corresponde ao código numérico da UF. A quantidade de municípios por UF será apresentada sem supor que os estados devam ter quantidades iguais.


In [ ]:
STATE_IDS = {
    "RO": "11",
    "AC": "12",
    "AM": "13",
    "RR": "14",
    "PA": "15",
    "AP": "16",
    "TO": "17",
    "MA": "21",
    "PI": "22",
    "CE": "23",
    "RN": "24",
    "PB": "25",
    "PE": "26",
    "AL": "27",
    "SE": "28",
    "BA": "29",
    "MG": "31",
    "ES": "32",
    "RJ": "33",
    "SP": "35",
    "PR": "41",
    "SC": "42",
    "RS": "43",
    "MS": "50",
    "MT": "51",
    "GO": "52",
    "DF": "53",
}

state_records = []

for state_code, state_id in STATE_IDS.items():

    state_records.append((state_code, state_id))

state_reference = spark.createDataFrame(
    state_records, "state_code string, expected_state_id string"
)
actual_states = population_view.select("state_code").distinct()

missing_states = state_reference.select("state_code").join(actual_states, "state_code", "left_anti")
unexpected_states = actual_states.join(
    state_reference.select("state_code"), "state_code", "left_anti"
)

state_check = population_view.join(state_reference, "state_code", "left")
state_issues = state_check.filter(
    F.col("expected_state_id").isNull() | ~F.col("state_id").eqNullSafe(F.col("expected_state_id"))
)

repeated_ids = (
    population_view.filter(F.col("municipality_id").isNotNull())
    .groupBy("municipality_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_id_count = repeated_ids.count()
missing_state_count = missing_states.count()

unexpected_state_count = unexpected_states.count()
state_issue_count = state_issues.count()

population_view.groupBy("state_code").count().orderBy("state_code").show(30, truncate=False)

print(f"IDs municipais repetidos: {duplicate_id_count}")
repeated_ids.show(truncate=False)

print(f"UFs ausentes: {missing_state_count}; UFs inesperadas: {unexpected_state_count}")
missing_states.show(truncate=False)
unexpected_states.show(truncate=False)

print(f"Divergências entre sigla e código da UF: {state_issue_count}")
state_issues.select("municipality_name", "state_code", "state_id", "expected_state_id").show(
    truncate=False
)


## 8. Recorte das capitais

A referência vem da Bronze de municípios, limitada às capitais. A junção é feita pelo código IBGE reconstruído, preservando o resultado mesmo quando a população não for encontrada.

Nomes e UFs são comparados, e uma capital sem população ou com várias correspondências é apresentada como ponto de investigação. Como a fonte de população cobre o país inteiro, seus milhares de registros fora desse recorte não são erros.


In [ ]:
capital_reference = municipalities.select(
    F.col("id").alias("capital_id"),
    F.col("nome").alias("capital_name"),
    F.col("state_code").alias("capital_state_code"),
).distinct()

capital_count = capital_reference.count()

capital_population = capital_reference.join(
    population_view,
    capital_reference["capital_id"] == population_view["municipality_id"],
    "left",
).cache()

name_matches = F.upper(F.trim(F.col("capital_name"))).eqNullSafe(
    F.upper(F.trim(F.col("municipality_name")))
)
state_matches = F.col("capital_state_code").eqNullSafe(F.col("state_code"))

capital_issues = capital_population.filter(
    F.col("municipality_id").isNull()
    | F.col("population_estimate").isNull()
    | (F.col("population_estimate") <= 0)
    | ~name_matches
    | ~state_matches
)

repeated_capital_matches = (
    capital_population.groupBy("capital_id").count().filter(F.col("count") > 1)
)
capital_issue_count = capital_issues.count()
repeated_capital_count = repeated_capital_matches.count()

print(f"Capitais na referência: {capital_count}")
print(f"Capitais com divergência ou população ausente/inválida: {capital_issue_count}")
print(f"Capitais com múltiplas correspondências: {repeated_capital_count}")

capital_issues.select(
    "capital_name", "capital_state_code", "municipality_name", "state_code", "population_estimate"
).show(truncate=False)
repeated_capital_matches.show(truncate=False)
capital_population.select(
    "capital_id", "capital_name", "capital_state_code", "population_estimate"
).orderBy(F.desc("population_estimate")).show(30, truncate=False)


## 9. Estatística descritiva

Calculamos os resumos separadamente para todos os registros com população válida e para as capitais. Os grupos se sobrepõem: capitais são um subconjunto, não uma segunda amostra independente.

A soma corresponde aos registros da planilha usados na análise. Não a apresentamos como uma validação independente do total oficial do país. Assimetrias e valores extremos serão interpretados no contexto do tamanho dos municípios.


In [ ]:
valid_population = population_view.filter(
    F.col("population_estimate").isNotNull() & (F.col("population_estimate") > 0)
)
valid_capitals = capital_population.filter(
    F.col("population_estimate").isNotNull() & (F.col("population_estimate") > 0)
)
valid_population_count = valid_population.count()
valid_capital_count = valid_capitals.count()

print("Todos os municípios com população válida:")
valid_population.select("population_estimate").summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
).show(truncate=False)

valid_population.agg(
    F.sum("population_estimate").alias("population_sum"),
    F.skewness("population_estimate").alias("skewness"),
    F.kurtosis("population_estimate").alias("excess_kurtosis"),
).show(truncate=False)

print("Capitais com população válida:")
valid_capitals.select("population_estimate").summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
).show(truncate=False)

valid_population.groupBy("state_code").agg(
    F.count("*").alias("municipality_count"),
    F.sum("population_estimate").alias("population_sum"),
).orderBy(F.desc("population_sum")).show(30, truncate=False)


## 10. Valores extremos pela regra do IQR

Aplicamos a regra da aula ao conjunto de municípios: valores fora de `Q1 - 1,5 × IQR` e `Q3 + 1,5 × IQR` são sinalizados. Isso não significa que estejam errados: municípios grandes podem ser observações legítimas, e a população tende a ser assimétrica.

Os registros sinalizados permanecem na análise e na Bronze. Não fazemos remoção, winsorização ou imputação. A presença de extremos estatísticos não entra automaticamente como falha no resumo de qualidade.


In [ ]:
iqr_outlier_count = 0

if valid_population_count > 0:
    q1, q3 = valid_population.approxQuantile("population_estimate", [0.25, 0.75], 0.0)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    iqr_outliers = valid_population.filter(
        (F.col("population_estimate") < lower_bound) | (F.col("population_estimate") > upper_bound)
    )

    iqr_outlier_count = iqr_outliers.count()

    print(f"Q1: {q1:,.0f}; Q3: {q3:,.0f}; IQR: {iqr:,.0f}")
    print(f"Limites estatísticos: {lower_bound:,.0f} a {upper_bound:,.0f}")

    print(f"Municípios sinalizados: {iqr_outlier_count} de {valid_population_count}")
    iqr_outliers.select("municipality_name", "state_code", "population_estimate").orderBy(
        F.desc("population_estimate")
    ).show(15, truncate=False)
else:
    print("Não há populações válidas para calcular os limites do IQR.")


## 11. Distribuições e dispersão

O primeiro gráfico compara a população na escala original com `ln(1 + população)`, apenas para visualização. A transformação não é gravada nem substitui a medida original.

O boxplot usa escala vertical logarítmica para mostrar valores muito diferentes. Os dois grupos se sobrepõem e a comparação é descritiva. São coletadas somente as poucas milhares de estimativas desta tabela, sem carregar dados meteorológicos no Python.


In [ ]:
plot_records = valid_population.select(
    "population_estimate", F.log1p("population_estimate").alias("log_population")
).collect()

population_values = []
log_population_values = []

for record in plot_records:

    population_values.append(record["population_estimate"])
    log_population_values.append(record["log_population"])

capital_records = valid_capitals.select("population_estimate").collect()
capital_values = []

for record in capital_records:

    capital_values.append(record["population_estimate"])

if population_values:
    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(population_values, bins=35, color="#2d223a", edgecolor="white")
    axes[0].set_title("População municipal — escala original")
    axes[0].set_xlabel("População estimada (habitantes)")
    axes[0].set_ylabel("Quantidade de municípios")
    axes[0].ticklabel_format(axis="x", style="sci", scilimits=(0, 0))
    axes[1].hist(log_population_values, bins=35, color="#2d223a", edgecolor="white")
    axes[1].set_title("População municipal — transformação visual")
    axes[1].set_xlabel("ln(1 + população estimada)")
    axes[1].set_ylabel("Quantidade de municípios")
    figure.tight_layout()
    plt.show()

if population_values and capital_values:
    figure, axis = plt.subplots(figsize=(8, 5))
    axis.boxplot(
        [population_values, capital_values], tick_labels=["Todos os municípios", "Capitais"]
    )
    axis.set_yscale("log")
    axis.set_title("Dispersão da população — grupos sobrepostos")
    axis.set_ylabel("População estimada (habitantes; escala log)")
    axis.grid(axis="y", color="lightgray", alpha=0.6)
    figure.tight_layout()
    plt.show()


## 12. População das capitais

Este gráfico é restrito às capitais identificadas pela chave IBGE. O eixo usa milhões de habitantes. Uma capital maior ou menor não é, por isso, melhor destino de viagem; aqui descrevemos apenas o tamanho populacional.


In [ ]:
ranking_records = (
    valid_capitals.select("capital_name", "capital_state_code", "population_estimate")
    .orderBy(F.desc("population_estimate"))
    .collect()
)
capital_labels = []
capital_millions = []

for record in ranking_records:

    capital_labels.append(f"{record['capital_name']} ({record['capital_state_code']})")
    capital_millions.append(record["population_estimate"] / 1_000_000)

if ranking_records:
    figure, axis = plt.subplots(figsize=(10, 10))
    axis.barh(capital_labels, capital_millions, color="#2d223a")
    axis.invert_yaxis()
    axis.set_title("População estimada das capitais — referência 2025")
    axis.set_xlabel("População estimada (milhões de habitantes)")
    axis.set_axisbelow(True)
    axis.grid(axis="x", color="lightgray", alpha=0.6)
    figure.tight_layout()
    plt.show()
else:
    print("Nenhuma capital com população válida foi encontrada.")


## 13. Síntese e decisões para a Silver

A síntese separa problemas de estrutura/qualidade dos títulos e notas legítimos e dos extremos estatísticos. As contagens podem se sobrepor.

**Encaminhamentos:** formalizar a extração dos registros de municípios; preservar a referência temporal e as notas; construir a chave de sete dígitos sem perder zeros; integrar as capitais pelo ID e investigar divergências. Nenhuma população é alterada com base nas notas de rodapé ou na regra do IQR nesta EDA.


In [ ]:
checks = [
    ("Linhas de origem duplicadas", source_duplicate_count),
    ("JSONs não interpretados", unparsed_row_count),
    ("Linhas com estrutura inesperada", unexpected_shape_count),
    ("Células essenciais ausentes", missing_cell_count),
    ("Chaves municipais inválidas", invalid_code_count),
    ("Populações inválidas ou não positivas", invalid_population_count),
    ("IDs municipais repetidos", duplicate_id_count),
    ("UFs ausentes", missing_state_count),
    ("UFs inesperadas", unexpected_state_count),
    ("Divergências entre sigla e código da UF", state_issue_count),
    ("Diferença em relação às 27 capitais esperadas", abs(capital_count - 27)),
    ("Capitais com divergência ou população ausente/inválida", capital_issue_count),
    ("Capitais com múltiplas correspondências", repeated_capital_count),
]

print(f"Linhas originais preservadas: {raw_row_count}")
print(f"Candidatos a municípios: {municipality_row_count}")
print(f"Linhas fora da tabela municipal: {nonmunicipal_row_count}")
print(f"Municípios com população válida: {valid_population_count}")
print(f"Registros de capitais com população válida: {valid_capital_count}")
print(f"Extremos estatísticos pelo IQR, mantidos na análise: {iqr_outlier_count}")

has_issues = False

for check_name, issue_count in checks:

    print(f"{check_name}: {issue_count}")

    if issue_count > 0:
        has_issues = True

if has_issues:
    print("Há pontos para investigar antes de definir os tratamentos da Silver.")
else:
    print("Nenhum problema foi encontrado nas verificações implementadas.")

print("Títulos, cabeçalho, notas e linhas vazias permanecem preservados na Bronze.")


## 14. Encerramento

As tabelas interpretadas e os gráficos são resultados de exploração. A Bronze não é sobrescrita e os dados não são publicados no banco.


In [ ]:
capital_population.unpersist()
population_view.unpersist()
parsed_rows.unpersist()
population_rows.unpersist()
spark.stop()
